# 05 Graph features

**Owner:** Person C. The first cell stops when the gate is not a clear yes.

**Inputs:** `outputs/graph_gate.json`, `data/processed/features.parquet`. An edge list, if that is the kind, goes in `data/processed/edges.csv` with `src` and `dst` entity ids.

**Outputs:** either `outputs/graph_skipped.md`, or `outputs/tables/graph_edges.csv`, `data/processed/features_with_graph.parquet`, and `outputs/graph_model_comparison.csv`.

**Stop rule:** a shared id alone is an entity key, not a graph. Feature-space kNN is not built. Neighbour label rates are computed only when the table already has one row per node; otherwise this notebook attaches degree only and says so. One XGBoost, validation only, runs when the frozen split and a target already exist.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.paths import (
    RAW,
    PROCESSED,
    OUTPUTS,
    TABLES,
    ANALYSIS_PATH,
    FEATURES_PATH,
    MODEL_MATRIX_PATH,
    ROLES_PATH,
    CONTRACT_PATH,
    GATE_PATH,
    FEATURE_LOG_PATH,
    ALIAS_PATH,
    ANSWERS_PATH,
    COMPARISON_PATH,
    BUNDLE_PATH,
    ensure_dirs,
)

ensure_dirs()
print("root", ROOT)


In [ ]:
import json

import numpy as np
import pandas as pd

from src import io as sio
from src.graph_gate import (
    edge_table,
    edges_from_coordinates,
    encode_edge_list,
    gate_is_yes,
    graph_features,
    read_gate,
)
from src.models import compare, design_matrix
from src.roles import columns_with_role, read_roles
from src.split import load_split, split_exists

gate = read_gate()
if not gate_is_yes(gate):
    message = "Graph features skipped.\n\n" + (gate.get("reason") or "The gate is not yes.")
    (OUTPUTS / "graph_skipped.md").write_text(message + "\n", encoding="utf-8")
    print(message)
elif not FEATURES_PATH.exists() and not FEATURES_PATH.with_suffix(".csv").exists():
    print("STOP: features.parquet is not ready.")
else:
    kind = gate.get("kind")
    frame = sio.read_frame(FEATURES_PATH).reset_index(drop=True)
    roles = read_roles()
    id_columns = [column for column in columns_with_role(roles, "id") if column in frame.columns]
    id_column = id_columns[0] if id_columns else None
    src = dst = None
    n_nodes = None
    node_ids = None
    if kind == "shared_id":
        print("STOP: a shared id is the entity key. Set kind to coordinates or edge_list, or set native_graph to false.")
    elif kind == "coordinates":
        geo = [column for column in columns_with_role(roles, "geo") if column in frame.columns]
        lat = next((column for column in geo if "lat" in column.lower()), None)
        lon = next((column for column in geo if "lon" in column.lower() or "lng" in column.lower()), None)
        if lat is None or lon is None:
            print("STOP: tag latitude and longitude with the geo role.")
        elif id_column is None:
            print("STOP: a coordinate graph needs an id column so rows can be collapsed to entities.")
        else:
            nodes = (
                frame.assign(**{id_column: frame[id_column].map(lambda value: "" if pd.isna(value) else str(value).strip())})
                .groupby(id_column, dropna=False)[[lat, lon]]
                .median(numeric_only=True)
                .reset_index()
            )
            packed = edges_from_coordinates(nodes[lat], nodes[lon])
            edge_table(packed, nodes[id_column]).to_csv(TABLES / "graph_edges.csv", index=False)
            print(
                "nodes", len(nodes),
                "edges", packed["n_edges"],
                "components", packed["n_components"],
                "sigma_km", packed["sigma_km"],
            )
            src, dst = packed["src"], packed["dst"]
            n_nodes = len(nodes)
            node_ids = nodes[id_column].astype(str)
    elif kind == "edge_list":
        edge_path = PROCESSED / "edges.csv"
        if id_column is None or not edge_path.exists():
            print("STOP: need an id column and data/processed/edges.csv with src and dst.")
        else:
            edges = pd.read_csv(edge_path)
            node_ids = pd.Series(frame[id_column].map(lambda value: "" if pd.isna(value) else str(value).strip()).unique())
            src, dst, dropped = encode_edge_list(edges, node_ids)
            print("edges kept", len(src), "dropped", dropped)
            n_nodes = len(node_ids)
            pd.DataFrame({"src": src, "dst": dst}).to_csv(TABLES / "graph_edges.csv", index=False)
    else:
        print("STOP: set graph_kind to coordinates or edge_list.")
    if src is not None:
        features = graph_features(src, dst, n_nodes)
        features.insert(0, id_column, node_ids.to_numpy())
        keyed = frame.copy()
        keyed[id_column] = keyed[id_column].map(lambda value: "" if pd.isna(value) else str(value).strip())
        graph_columns = [column for column in features.columns if column.startswith("g_")]
        attached = keyed[[id_column]].merge(features[[id_column] + graph_columns], on=id_column, how="left")
        if len(attached) != len(keyed):
            print("STOP: graph merge changed the row count. The entity id is duplicated on the node table.")
        else:
            enriched = pd.concat([keyed.reset_index(drop=True), attached[graph_columns].reset_index(drop=True)], axis=1)
            written = sio.save_frame(enriched, PROCESSED / "features_with_graph.parquet")
            print("wrote", written, "columns", graph_columns)
            print("Degree is attached. Neighbour label rates were not, because this table may have many rows per node.")
            target = next((column for column in columns_with_role(roles, "target") if column in enriched.columns), None)
            if target and split_exists():
                split = load_split()
                blocked = set(columns_with_role(roles, "id"))
                base_columns = [
                    column
                    for column in enriched.columns
                    if column != target and column not in blocked and not str(column).startswith(("entity_", "g_"))
                ]
                with_graph = base_columns + graph_columns
                y_train = enriched.iloc[split["train"]][target]
                y_val = enriched.iloc[split["val"]][target]
                rows = []
                for label, columns in (("without_graph", base_columns), ("with_graph", with_graph)):
                    built = design_matrix(enriched, columns, split["train"], split["val"])
                    report, _chosen = compare(
                        built["X_train"], y_train, built["X_other"], y_val, families=("xgboost",)
                    )
                    report.insert(0, "features", label)
                    rows.append(report)
                graph_report = pd.concat(rows, ignore_index=True)
                graph_report.to_csv(OUTPUTS / "graph_model_comparison.csv", index=False)
                print(graph_report.to_string(index=False))
                print("Test rows were not scored.")
            else:
                print("XGBoost comparison skipped until a target and the frozen split both exist.")
